# Miner Tips 04: Net2Net Widen Then Quantize

Net2Net-style widening creates a larger student that initially behaves like the smaller model. Then you quantize/distill the wider student.

Why this can help:

```text
small dense model
        |
function-preserving widen
        v
larger dense student with similar behavior
        |
low-bit quantization + distillation
        v
binary/ternary artifact with more low-bit capacity
```

The widening step is not magic. It just gives the low-bit student more rows/columns to spend while starting from a sensible function.

In [ ]:
import numpy as np

rng = np.random.default_rng(11)

# A tiny two-layer MLP: y = W2 * tanh(W1 * x)
in_dim = 4
hidden = 3
out_dim = 2
W1 = rng.normal(size=(hidden, in_dim)).astype(np.float32)
W2 = rng.normal(size=(out_dim, hidden)).astype(np.float32)

def mlp(x, W1, W2):
    return np.tanh(x @ W1.T) @ W2.T

x = rng.normal(size=(5, in_dim)).astype(np.float32)
y = mlp(x, W1, W2)
print("original output shape:", y.shape)

In [ ]:
def widen_hidden_by_duplication(W1, W2, copies=2):
    """Duplicate hidden units while preserving the function.

    If one hidden unit is duplicated twice, each duplicate gets half of the
    outgoing weight. Their sum equals the original unit's contribution.
    """
    W1_wide = np.repeat(W1, copies, axis=0)
    W2_wide = np.repeat(W2 / copies, copies, axis=1)
    return W1_wide, W2_wide

W1_wide, W2_wide = widen_hidden_by_duplication(W1, W2, copies=3)
y_wide = mlp(x, W1_wide, W2_wide)

print("wide hidden size:", W1_wide.shape[0])
print("max output difference after widening:", np.max(np.abs(y - y_wide)))

In [ ]:
def ternary_np(W):
    """Toy ternary projection for demonstration."""
    scale = np.mean(np.abs(W)) + 1e-8
    threshold = 0.7 * scale
    return np.where(np.abs(W) >= threshold, np.sign(W) * scale, 0.0).astype(np.float32)

W1_q = ternary_np(W1_wide)
W2_q = ternary_np(W2_wide)
y_q = mlp(x, W1_q, W2_q)

print("mse after widening only:", float(np.mean((y - y_wide) ** 2)))
print("mse after widening + toy ternary:", float(np.mean((y - y_q) ** 2)))

## Real-model translation

For transformer models, the same idea appears in places like:

- duplicate hidden channels in MLP projections;
- split outgoing weights so duplicated channels sum to the old contribution;
- keep normalization behavior stable;
- be careful with tied embeddings and language-model heads;
- after widening, run layerwise distillation before final PPL checks.

A simple rule of thumb for low-bit students is to search for a capacity point where extra low-bit parameters offset binary/ternary error without making the artifact too large.